In [0]:
WITH params AS (
  SELECT 30 AS p_window_days
),
bronze_base AS (
  SELECT
    try_to_timestamp(ingested_at_utc) AS ingested_ts_utc,
    weakness_id,
    title,
    description,
    extended_description,
    potential_mitigations
  FROM 3dt2ndteam5.cwe.cwe_weaknesses
),
scoped AS (
  SELECT
    ingested_ts_utc,
    weakness_id,
    title,
    description,
    extended_description,
    potential_mitigations
  FROM bronze_base
  CROSS JOIN params p
  WHERE ingested_ts_utc IS NOT NULL
    AND ingested_ts_utc >= timestampadd(DAY, -p.p_window_days, current_timestamp())
),
derived AS (
  SELECT
    ingested_ts_utc,
    CASE
      WHEN upper(trim(COALESCE(title, ''))) LIKE 'DEPRECATED:%' THEN 1
      ELSE 0
    END AS is_deprecated,
    trim(
      regexp_replace(
        concat_ws(
          ' ',
          concat('CWE-', COALESCE(weakness_id, '')),
          trim(regexp_replace(COALESCE(title, ''), '(?i)^\\s*DEPRECATED:\\s*', '')),
          COALESCE(description, ''),
          COALESCE(extended_description, ''),
          COALESCE(potential_mitigations, '')
        ),
        '\\s+',
        ' '
      )
    ) AS derived_search_text
  FROM scoped
),
silver_like_daily AS (
  SELECT
    date(ingested_ts_utc) AS metric_date_utc,
    date(from_utc_timestamp(ingested_ts_utc, 'Asia/Seoul')) AS metric_date_kst,
    COUNT(*) AS evaluated_rows,
    SUM(
      CASE
        WHEN derived_search_text IS NULL OR trim(derived_search_text) = '' THEN 1
        ELSE 0
      END
    ) AS empty_search_text_rows
  FROM derived
  WHERE is_deprecated = 0
  GROUP BY
    date(ingested_ts_utc),
    date(from_utc_timestamp(ingested_ts_utc, 'Asia/Seoul'))
)
SELECT
  metric_date_utc,
  metric_date_kst,
  evaluated_rows,
  empty_search_text_rows,
  ROUND(
    CASE
      WHEN evaluated_rows = 0 THEN 0.0
      ELSE (empty_search_text_rows * 100.0) / evaluated_rows
    END,
    4
  ) AS empty_search_text_ratio_pct
FROM silver_like_daily
ORDER BY metric_date_utc;
